In [3]:
from pathlib import Path

import pandas as pd

PATH_TRAIN_VOCAB = Path("../data/playlist/training_vocab.parquet")

training_vocab = pd.read_parquet(PATH_TRAIN_VOCAB)
print(f"Loaded training_vocab: {len(training_vocab):,} tracks")

Loaded training_vocab: 27,176,693 tracks


# Parameters
Edit this cell to explore different configurations.

In [4]:
CMIN       = 10   # min global playlist count to include a track
EMBED_DIM  = 128   # embedding dimension
BATCH_SIZE = 128_000
W          = 5     # context window half-width
K          = 15    # negatives per positive

# Vocab size vs CMIN
Sweep over CMIN thresholds to understand the vocab/memory tradeoff.

In [5]:
GiB = 1 / 1_073_741_824

print(f"{'CMIN':>6}  {'vocab':>12}  {'embed (GB)':>10}  {'embed+optim (GB)':>16}")
print("-" * 52)
for cmin in [2, 5, 10, 20, 50, 100, 200, 500, 1000]:
    vs = len(training_vocab[training_vocab["playlist_count"] >= cmin])
    embed_gb = 2 * vs * EMBED_DIM * 4 * GiB
    total_gb = 3 * embed_gb  # embed + optimizer state (exp_avg + exp_avg_sq)
    marker = "  <-- selected" if cmin == CMIN else ""
    print(f"  {cmin:>4}  {vs:>12,}  {embed_gb:>10.2f}  {total_gb:>16.2f}{marker}")

  CMIN         vocab  embed (GB)  embed+optim (GB)
----------------------------------------------------
     2    27,176,693       25.92             77.75
     5    14,094,489       13.44             40.32
    10     8,764,708        8.36             25.08  <-- selected
    20     5,412,049        5.16             15.48
    50     2,767,637        2.64              7.92
   100     1,600,623        1.53              4.58
   200       888,775        0.85              2.54
   500       389,254        0.37              1.11
  1000       202,007        0.19              0.58


# Detailed memory breakdown
Full GPU memory estimate for the selected configuration.

In [6]:
vocab_size = len(training_vocab[training_vocab["playlist_count"] >= CMIN])
B, D = BATCH_SIZE, EMBED_DIM
GiB = 1 / 1_073_741_824
MiB = 1 / 1_048_576

embed    = 2 * vocab_size * D * 4   # both embedding tables (in + out)
optim    = 2 * embed                 # SparseAdam: exp_avg + exp_avg_sq
                                     # initialised as dense tensors — same size as embed
weights_ = vocab_size * 4            # neg-sampling weights on GPU
act_fwd  = (2 + K) * B * D * 4      # ecenter (B×D) + econtext (B×D) + enegative (B×K×D)
act_bwd  = act_fwd                   # sparse grads upper bound ≈ forward activations
total    = embed + optim + weights_ + act_fwd + act_bwd

print(f"Configuration")
print(f"  CMIN={CMIN}  EMBED_DIM={D}  BATCH_SIZE={B:,}  W={W}  K={K}")
print(f"  Vocab size : {vocab_size:,}")
print()
print(f"GPU memory breakdown")
print(f"  Embedding tables    : {embed    * GiB:>6.2f} GB  (2 tables × {vocab_size:,} × {D} × fp32)")
print(f"  Optimizer state     : {optim    * GiB:>6.2f} GB  (SparseAdam exp_avg + exp_avg_sq, dense after warmup)")
print(f"  Neg. sample weights : {weights_ * MiB:>6.1f} MB")
print(f"  Activations fwd     : {act_fwd  * MiB:>6.1f} MB  ({2+K} tensors × {B:,} × {D})")
print(f"  Activations bwd     : {act_bwd  * MiB:>6.1f} MB  (sparse grad upper bound)")
print()
print(f"  Total estimate      : {total * GiB:>6.2f} GB")
print()

# Cloud instance reference
INSTANCES = [
    ("2060",         6),
    ("T4",          16),
    ("A10G",        24),
    ("A100 40GB",   40),
    ("V100 32GB",   32),
    ("A100 80GB",   80),
    ("H100 80GB",   80),
]
total_gb = total * GiB
print(f"  {'Instance':<14}  {'VRAM (GB)':>9}  {'Headroom (GB)':>13}  {'':>8}")
print(f"  {'-'*50}")
for name, vram_gb in INSTANCES:
    headroom = vram_gb - total_gb
    flag = "OK" if headroom > 0 else "OOM RISK"
    print(f"  {name:<14}  {vram_gb:>9.0f}  {headroom:>+13.1f}  {flag}")

Configuration
  CMIN=10  EMBED_DIM=128  BATCH_SIZE=128,000  W=5  K=15
  Vocab size : 8,764,708

GPU memory breakdown
  Embedding tables    :   8.36 GB  (2 tables × 8,764,708 × 128 × fp32)
  Optimizer state     :  16.72 GB  (SparseAdam exp_avg + exp_avg_sq, dense after warmup)
  Neg. sample weights :   33.4 MB
  Activations fwd     : 1062.5 MB  (17 tensors × 128,000 × 128)
  Activations bwd     : 1062.5 MB  (sparse grad upper bound)

  Total estimate      :  27.18 GB

  Instance        VRAM (GB)  Headroom (GB)          
  --------------------------------------------------
  2060                    6          -21.2  OOM RISK
  T4                     16          -11.2  OOM RISK
  A10G                   24           -3.2  OOM RISK
  A100 40GB              40          +12.8  OK
  V100 32GB              32           +4.8  OK
  A100 80GB              80          +52.8  OK
  H100 80GB              80          +52.8  OK


# Training pairs estimate
Rough estimate of pairs per epoch given W, based on the interactions that survive CMIN filtering.

In [14]:
# Total interactions in the full dataset surviving CMIN
# (lower bound: each interaction is a (playlist, track) pair; pairs per interaction ≈ 2*W)
total_interactions = training_vocab[training_vocab["playlist_count"] >= CMIN]["playlist_count"].sum()
pairs_estimate = total_interactions * 2 * W
steps_per_epoch = pairs_estimate / BATCH_SIZE

print(f"CMIN={CMIN}  W={W}  BATCH_SIZE={B:,}")
print()
print(f"  Total interactions (full dataset) : {total_interactions:>15,}")
print(f"  Pairs/epoch estimate (× 2W)       : {pairs_estimate:>15,}")
print(f"  Steps/epoch                       : {steps_per_epoch:>15,.0f}")
print()
print("Note: 'interactions' = sum of playlist_count across vocab, not distinct (playlist, track) pairs.")
print("Pairs/epoch is an overestimate; actual count depends on subsampling and playlist length distribution.")

CMIN=10  W=5  BATCH_SIZE=32,000

  Total interactions (full dataset) :   1,580,219,217
  Pairs/epoch estimate (× 2W)       :  15,802,192,170
  Steps/epoch                       :         493,819

Note: 'interactions' = sum of playlist_count across vocab, not distinct (playlist, track) pairs.
Pairs/epoch is an overestimate; actual count depends on subsampling and playlist length distribution.
